# LAVKA Incremental Loader TEST

Тестовый режим:
- сначала загружаем и проверяем данные;
- все сохранения выполняются только в последней ячейке `COMMIT`.


In [ ]:
import os
import re
from datetime import datetime

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, LongType

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
path_components = notebook_path.split("/")
team_folder = path_components[2]

import sys
sys.path.append(
    f"/Workspace/eperfectstore-prod/{team_folder}/notebooks/eperfectstore-prod/e-com/COMMON_FUNCTIONS_AND_CONSTANTS_FOLDER/"
)
from common_functions_and_constants import *


In [ ]:
# Centralized configuration
CONFIG = {
    "sharepoint": {
        "po1_base_url": "https://pepsico.sharepoint.com/teams/RussiaSPO1CustomerCollaboration/",
        "wbd_base_url": "https://pepsico.sharepoint.com/teams/AzureCCplatform/",
    },
    "paths": {
        "po1_main": {
            "spo_dir": "Shared Documents/General/E-COM/Клиенты/2.Лавка/DOS отчет/DOS REP/Метрики cpfr/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_po1_test",
            "dbfs_scan": "/ecom/lavka_dos_po1_test",
        },
        "po1_archive": {
            "spo_dir": "Shared Documents/General/E-COM/Клиенты/2.Лавка/DOS отчет/DOS REP/Метрики cpfr/archiv",
            "dbfs_local": "/dbfs/ecom/lavka_dos_po1_test_archive",
            "dbfs_scan": "/ecom/lavka_dos_po1_test_archive",
        },
        "po1_kub": {
            "spo_dir": "Shared Documents/General/E-COM/Клиенты/2.Лавка/Для КУБ Лавка/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_po1_kub",
            "dbfs_scan": "/ecom/lavka_dos_po1_kub",
        },
        "po1_dictionaries": {
            "spo_dir": "Shared Documents/General/E-COM/Клиенты/2.Лавка/DICTIONARIES/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_po1_test/DICTIONARIES",
        },
        "wbd_ao": {
            "spo_dir": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_ao/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_wbd/",
            "dbfs_scan": "/ecom/lavka_dos_wbd/",
        },
        "wbd_orders": {
            "spo_dir": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_orders/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_wbd/metrics_orders/",
            "dbfs_scan": "/ecom/lavka_dos_wbd/metrics_orders/",
        },
        "wbd_osa": {
            "spo_dir": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_osa/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_wbd/metrics__osa/",
            "dbfs_scan": "/ecom/lavka_dos_wbd/metrics__osa/",
        },
        "wbd_prediction": {
            "spo_dir": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_prediction/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_wbd/metrics_prediction/",
            "dbfs_scan": "/ecom/lavka_dos_wbd/metrics_prediction/",
        },
        "wbd_sales_stock": {
            "spo_dir": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_sales_stock/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_wbd/metrics_sales_stock/",
            "dbfs_scan": "/ecom/lavka_dos_wbd/metrics_sales_stock/",
        },
        "wbd_directory": {
            "spo_dir": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/directory/",
            "dbfs_local": "/dbfs/ecom/lavka_dos_wbd/directory/",
        },
    },
    "targets": {
        "po1": {
            "table": "ECOM_ETL.LAVKA_PO1",
            "path": "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka/lavka_po1",
        },
        "po1_assort": {
            "table": "ECOM_ETL.LAVKA_PO1_ASSORT",
            "path": "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka/lavka_po1_assort",
        },
        "wbd_ao": {
            "table": "ECOM_ETL.LAVKA_WBD_METRICS_AO",
            "path": "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka/lavka_wbd/metrics_ao",
        },
        "wbd_orders": {
            "table": "ECOM_ETL.LAVKA_WBD_METRICS_ORDERS",
            "path": "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka/lavka_wbd/metrics_orders",
        },
        "wbd_osa": {
            "table": "ECOM_ETL.LAVKA_WBD_METRICS_OSA",
            "path": "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka/lavka_wbd/metrics_osa",
        },
        "wbd_prediction": {
            "table": "ECOM_ETL.LAVKA_WBD_METRICS_PREDICTION",
            "path": "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka/lavka_wbd/metrics_prediction",
        },
        "wbd_sales_stock": {
            "table": "ECOM_ETL.LAVKA_WBD_METRICS_SALES_STOCK",
            "path": "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka/lavka_wbd/metrics_sales_stock",
        },
        "wbd_directory": {
            "table": "ECOM_ETL.LAVKA_WBD_DIRECTORY",
            "path": "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka/lavka_wbd/directory",
        },
    },
}

CONFIG["cleanup_dirs"] = [
    CONFIG["paths"]["po1_main"]["dbfs_scan"],
    CONFIG["paths"]["po1_archive"]["dbfs_scan"],
    CONFIG["paths"]["po1_kub"]["dbfs_scan"],
    CONFIG["paths"]["wbd_ao"]["dbfs_scan"],
    CONFIG["paths"]["wbd_orders"]["dbfs_scan"],
    CONFIG["paths"]["wbd_osa"]["dbfs_scan"],
    CONFIG["paths"]["wbd_prediction"]["dbfs_scan"],
    CONFIG["paths"]["wbd_sales_stock"]["dbfs_scan"],
    CONFIG["paths"]["wbd_directory"]["dbfs_local"].replace("/dbfs", "dbfs:"),
]


In [ ]:
# Helpers
def log(msg: str):
    print(f"[{datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC] {msg}")


def normalize_col_name(name: str) -> str:
    s = re.sub(r"[^0-9a-zA-Z]+", "_", name.strip().lower())
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def normalize_spark_columns(df):
    out = df
    for c in out.columns:
        new_c = normalize_col_name(c)
        if new_c != c:
            out = out.withColumnRenamed(c, new_c)
    return out


def _parse_metric_date(file_path: str):
    name = os.path.splitext(os.path.basename(file_path))[0]
    m = re.search(r"\d{8}", name)
    if not m:
        return None
    raw = m.group()
    for fmt in ("%d%m%Y", "%Y%m%d"):
        try:
            return datetime.strptime(raw, fmt).date()
        except ValueError:
            pass
    return None


def table_exists(table_name: str) -> bool:
    try:
        spark.read.table(table_name).limit(1).count()
        return True
    except Exception:
        return False


def sync_sharepoint(base_url: str, spo_dir: str, dbfs_local: str):
    log(f"Sync: {spo_dir} -> {dbfs_local}")
    copy_from_spo(base_url, spo_dir, dbfs_local)


def list_new_files(dbfs_scan_dir: str, extension: str, target_table: str):
    files = []
    for f in dbutils.fs.ls(dbfs_scan_dir):
        if f.path.lower().endswith(extension.lower()):
            full_path = f.path.replace("dbfs:/", "/dbfs/")
            files.append((full_path, os.path.basename(full_path), _parse_metric_date(full_path)))

    if not files:
        return []

    files_schema = StructType([
        StructField("full_path", StringType(), False),
        StructField("_source_file", StringType(), False),
        StructField("_file_metric_date", DateType(), True),
    ])
    files_df = spark.createDataFrame(files, schema=files_schema)

    if not table_exists(target_table):
        return [r[0] for r in files_df.select("full_path").collect()]

    target_df = spark.read.table(target_table)
    target_cols = set(target_df.columns)

    if "_source_file" in target_cols:
        loaded_files = target_df.select("_source_file").distinct()
        new_df = files_df.join(loaded_files, on="_source_file", how="left_anti")
        return [r[0] for r in new_df.select("full_path").collect()]

    if "metric_date" in target_cols:
        max_metric_date = target_df.select(F.max("metric_date").alias("max_d")).collect()[0]["max_d"]
        if max_metric_date is not None:
            new_df = files_df.where(F.col("_file_metric_date") > F.lit(max_metric_date))
            return [r[0] for r in new_df.select("full_path").collect()]

    return [r[0] for r in files_df.select("full_path").collect()]


def read_po1_excels(paths):
    required_cols = ["Date", "City", "Supplier", "Item", "Item_Name", "Metric", "Value"]
    optional_col = "w"
    target_sheets = ["Лист1", "Sheet1"]
    dtype_map = {c: "string" for c in ["City", "Supplier", "Item", "Item_Name", "Metric", "Value"]}

    dfs = []
    for path in paths:
        file_metric_date = _parse_metric_date(path)
        for sheet in target_sheets:
            try:
                df = pd.read_excel(path, sheet_name=sheet, header=0)
            except Exception:
                continue

            if not set(required_cols).issubset(df.columns):
                continue

            cols_to_keep = required_cols + ([optional_col] if optional_col in df.columns else [])
            df = df[cols_to_keep]
            # TEST mode: keep raw source values (no type casting)

            df["metric_date"] = file_metric_date
            df["_source_file"] = os.path.basename(path)
            dfs.append(df)

    if not dfs:
        return None
    sdf = spark.createDataFrame(pd.concat(dfs, ignore_index=True))
    return normalize_spark_columns(sdf)


def read_csv_batch(paths):
    if not paths:
        return None
    dfs = []
    for path in paths:
        df = pd.read_csv(path, header=0)
        df["_source_file"] = os.path.basename(path)
        dfs.append(df)
    if not dfs:
        return None
    sdf = spark.createDataFrame(pd.concat(dfs, ignore_index=True))
    return normalize_spark_columns(sdf)


def preview_row_count(df, label: str):
    rows = df.count()
    log(f"{label} preview rows: {rows}")


def cleanup_staging_dirs(dbfs_dirs):
    log("Cleanup staging dirs (keep folders)")
    for d in dbfs_dirs:
        try:
            children = dbutils.fs.ls(d)
        except Exception as e:
            log(f"Skip list {d}: {e}")
            continue

        for ch in children:
            try:
                dbutils.fs.rm(ch.path, True)
            except Exception as e:
                log(f"Skip remove {ch.path}: {e}")

        log(f"Cleaned: {d}")



## Preview PO1 (без сохранения)


In [ ]:
po1_target = CONFIG["targets"]["po1"]

for key in ["po1_main", "po1_archive", "po1_kub"]:
    path_cfg = CONFIG["paths"][key]
    sync_sharepoint(CONFIG["sharepoint"]["po1_base_url"], path_cfg["spo_dir"], path_cfg["dbfs_local"])

po1_files = {
    key: list_new_files(CONFIG["paths"][key]["dbfs_scan"], ".xlsx", po1_target["table"])
    for key in ["po1_main", "po1_archive", "po1_kub"]
}
all_po1_files = po1_files["po1_main"] + po1_files["po1_archive"] + po1_files["po1_kub"]

log(f"PO1 new files total: {len(all_po1_files)}")
log(f"PO1 breakdown: main={len(po1_files['po1_main'])}, archive={len(po1_files['po1_archive'])}, kub={len(po1_files['po1_kub'])}")

po1_batch = read_po1_excels(all_po1_files)
if po1_batch is not None:
    po1_batch = po1_batch.withColumn("_loaded_at", F.current_timestamp())
    preview_row_count(po1_batch, "PO1")
    display(po1_batch.limit(50))
else:
    log("PO1: no new files")


## Preview PO1 Dictionaries (без сохранения)


In [ ]:
dict_path = CONFIG["paths"]["po1_dictionaries"]
dict_target = CONFIG["targets"]["po1_assort"]

sync_sharepoint(CONFIG["sharepoint"]["po1_base_url"], dict_path["spo_dir"], dict_path["dbfs_local"])

assort_lavka_pd_df = pd.read_excel(
    "/dbfs/ecom/lavka_dos_po1_test/DICTIONARIES/Assort Lavka.xlsx",
    sheet_name="Assort",
    header=0,
)
assort_lavka_df = normalize_spark_columns(spark.createDataFrame(assort_lavka_pd_df))
preview_row_count(assort_lavka_df, "PO1 assort")
display(assort_lavka_df.limit(50))


## Preview WBD Metrics (без сохранения)


In [ ]:
def preview_wbd_metric(path_cfg: dict, target_cfg: dict):
    sync_sharepoint(CONFIG["sharepoint"]["wbd_base_url"], path_cfg["spo_dir"], path_cfg["dbfs_local"])

    new_files = list_new_files(path_cfg["dbfs_scan"], ".csv", target_cfg["table"])
    log(f"{target_cfg['table']} new files: {len(new_files)}")
    if not new_files:
        return

    batch_df = read_csv_batch(new_files)
    if batch_df is None:
        log(f"{target_cfg['table']}: parsed empty batch")
        return

    # Parse metrics_version as date and drop invalid rows
    if "metrics_version" in batch_df.columns:
        before_rows = batch_df.count()
        batch_df = batch_df.withColumn("metrics_version", F.expr("try_cast(metrics_version as date)"))
        batch_df = batch_df.filter(F.col("metrics_version").isNotNull())
        after_rows = batch_df.count()
        log(f"{target_cfg['table']} filtered by metrics_version date: {before_rows} -> {after_rows}")

    batch_df = batch_df.withColumn("_loaded_at", F.current_timestamp())
    preview_row_count(batch_df, target_cfg['table'])
    display(batch_df.limit(20))


wbd_pairs = [
    ("wbd_ao", "wbd_ao"),
    ("wbd_orders", "wbd_orders"),
    ("wbd_osa", "wbd_osa"),
    ("wbd_prediction", "wbd_prediction"),
    ("wbd_sales_stock", "wbd_sales_stock"),
]

for path_key, target_key in wbd_pairs:
    preview_wbd_metric(CONFIG["paths"][path_key], CONFIG["targets"][target_key])


## Preview WBD Directory (без сохранения)


In [ ]:
wbd_dir_path = CONFIG["paths"]["wbd_directory"]
wbd_dir_target = CONFIG["targets"]["wbd_directory"]

sync_sharepoint(CONFIG["sharepoint"]["wbd_base_url"], wbd_dir_path["spo_dir"], wbd_dir_path["dbfs_local"])

products_df = pd.read_csv("/dbfs/ecom/lavka_dos_wbd/directory/products.csv")
products_df = normalize_spark_columns(spark.createDataFrame(products_df))
preview_row_count(products_df, "WBD directory")
display(products_df.limit(50))


## Write Disabled in TEST


In [ ]:
log("TEST notebook runs in read-only preview mode: no writes to target tables.")


## Cleanup (optional)

Опционально: очистка staging-папок после проверки preview.


In [ ]:
cleanup_staging_dirs(CONFIG["cleanup_dirs"])
log("Run finished")
